# **Mediapipe video analysis**
## Best run on GPU

# Imports

In [13]:
import os
import cv2
import subprocess
import pandas as pd
from google.cloud import storage
from collections import defaultdict
from IPython.display import Video, display
from google.colab.patches import cv2_imshow

# Set Up

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
%%capture
!pip install mediapipe
!wget -q -O efficientdet.tflite -q https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float16/1/efficientdet_lite0.tflite

In [16]:
!gcloud auth login # Authenticates your identity for general gcloud CLI command use e.g. gcloud storage cp, gcloud compute, gsutil
!gcloud auth application-default login # Authenticates your environment for API access e.g., storage.Client()

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=mhLBfOS6aNHpsg75llTbLKBLHe6jOz&prompt=consent&token_usage=remote&access_type=offline&code_challenge=ml2axYkehDFkcgkXjPGyoEdV_HqMAGmDPl5brOMgTd4&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0ATX87lPxsReJ4wfwqeKWcc7CU8V8M7LAlZhoSIwCD4EfVqiY8vH96BNQUadiDbX33HVbug

You are now logged in as [samiratra95@gmail.com].
Your current project 

In [17]:
client = storage.Client(project="brb-traffic")

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [18]:
base_dir = "/"
bucket_name = 'brb-traffic'
video_path='videos'
os.makedirs(video_path, exist_ok=True)

# Read Data

In [19]:
# Load the dataset
train = pd.read_csv(os.path.join(base_dir, '/content/Train.csv'))

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

(16076, 14)

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


In [20]:
ss = pd.read_csv(os.path.join(base_dir,'/content/SampleSubmission.csv'))
display(ss.shape,ss.head())

(880, 3)

,ID,Target,Target_Accuracy
0,time_segment_129_Norman Niles #1_congestion_en...,free flowing,free flowing
1,time_segment_130_Norman Niles #1_congestion_en...,heavy delay,heavy delay
2,time_segment_131_Norman Niles #1_congestion_en...,free flowing,free flowing
3,time_segment_132_Norman Niles #1_congestion_en...,heavy delay,heavy delay
4,time_segment_133_Norman Niles #1_congestion_en...,free flowing,free flowing


In [ ]:
# test = pd.read_csv(os.path.join(base_dir,'TestInputSegments.csv'))
# display(test.shape,test.head())

# Download files from a google cloud storage

In [21]:
blobs=train.videos.tolist()[12030:12070]
print(f"Number of blobs selected: {len(blobs)}")
display(blobs)

Number of blobs selected: 40


['normanniles3/normanniles3_2025-10-26-17-32-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-33-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-34-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-35-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-36-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-37-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-38-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-39-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-40-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-41-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-42-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-43-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-44-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-46-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-47-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-48-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-49-45.mp4',
 'normanniles3/normanniles3_2025-10-26-17-50-45.mp4',
 'normanniles3/normanniles3_

In [ ]:
from google.api_core.exceptions import NotFound

print(f"--- Debugging Blobs Variable ---")
print(f"Type of 'blobs' before loop: {type(blobs)}")
print(f"Content of 'blobs' (first 5): {blobs[:5]}")
print(f"----------------------------------")

# --- Existing loop for downloading files from the 'blobs' list ---
for blob_name in blobs:
    # Ensure blob_name is a string before proceeding
    if not isinstance(blob_name, str):
        print(f"❌ Error: Expected string for blob_name, but got {type(blob_name)}. Skipping.")
        continue

    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print(f"Attempting to download blob: '{blob_name}' to '{local_path}'") # Added print for clarity
    try:
        blob.download_to_filename(local_path)
        print("✅ Downloaded to:", local_path)
    except NotFound:
        print(f"❌ Error: '{blob_name}' not found in bucket '{bucket_name}'. Skipping.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while downloading '{blob_name}': {e}")

In [23]:
import re

# Set this to a specific video filename (e.g., 'normanniles3_2025-10-26-17-50-45.mp4')
# to download that particular file. Leave as None to skip.
specific_video_to_download = "normanniles3_2025-10-26-17-50-45.mp4" # Uncomment and change to download a specific file

# --- New section for downloading a specific file by name ---
if specific_video_to_download:
    print(f"\n--- Attempting to download specific file: {specific_video_to_download} ---")
    specific_blob_path = None

    # Try to find the full path from the train DataFrame
    # This part assumes 'train' DataFrame is available and has 'videos' column
    if 'train' in globals() and not train.empty:
        matched_rows = train[train['videos'].str.contains(specific_video_to_download, na=False, regex=False)]
        if not matched_rows.empty:
            # Take the first match, assuming filename is unique enough
            specific_blob_path = matched_rows['videos'].iloc[0]
        else:
            # If not found in train, assume it's directly in a camera folder (e.g. normanniles3/filename)
            camera_id_match = re.match(r'([a-zA-Z0-9]+)', specific_video_to_download)
            if camera_id_match:
                camera_id = camera_id_match.group(1)
                specific_blob_path = f"{camera_id}/{specific_video_to_download}"
            else:
                # Fallback: assume the specific_video_to_download is already the full blob path if no camera_id can be extracted
                specific_blob_path = specific_video_to_download
    else:
        # If train DataFrame is not available, assume the specific_video_to_download is the full blob path
        specific_blob_path = specific_video_to_download

    if specific_blob_path:
        blob = client.bucket(bucket_name).blob(specific_blob_path)
        file_name = os.path.basename(specific_video_to_download)
        local_path = os.path.join(video_path, file_name)

        print(f"Attempting to download specific blob: '{specific_blob_path}' to '{local_path}'")
        try:
            blob.download_to_filename(local_path)
            print(f"✅ Downloaded specific file to: {local_path}")
        except NotFound:
            print(f"❌ Error: Specific file '{specific_blob_path}' not found in bucket '{bucket_name}'.")
        except Exception as e:
            print(f"❌ An unexpected error occurred while downloading specific file '{specific_blob_path}': {e}")
    else:
        print(f"❌ Could not determine full blob path for '{specific_video_to_download}'.")


--- Attempting to download specific file: normanniles3_2025-10-26-17-50-45.mp4 ---
Attempting to download specific blob: 'normanniles3/normanniles3_2025-10-26-17-50-45.mp4' to 'videos/normanniles3_2025-10-26-17-50-45.mp4'
✅ Downloaded specific file to: videos/normanniles3_2025-10-26-17-50-45.mp4


# Delete videos

In [ ]:
import os
import shutil # Import shutil, though we'll adjust its usage

# Get the list of filenames that should exist in video_path
relevant_filenames = {os.path.basename(blob_path) for blob_path in blobs}

# Get the list of actual files in the video_path directory
current_video_files = os.listdir(video_path)

print(f"Number of relevant files (from blobs): {len(relevant_filenames)}")
print(f"Number of files currently in video_path: {len(current_video_files)}")

# Iterate through current files and delete those that ARE relevant
print("Deleting relevant videos from the directory...")
for filename in current_video_files:
    file_to_delete = os.path.join(video_path, filename)
    is_file = os.path.isfile(file_to_delete)
    is_mp4 = filename.lower().endswith('.mp4')

    print(f"Checking {filename}: Is relevant={filename in relevant_filenames}, Is file={is_file}, Is MP4={is_mp4}")

    if filename in relevant_filenames: # Condition inverted to delete relevant files
        if is_file and is_mp4:
            os.remove(file_to_delete)
            print(f"Deleted relevant MP4 file: {file_to_delete}")
    # else:
        # print(f"Keeping irrelevant file: {filename}") # Uncomment to see which files are kept

# Play some videos downloaded earlier

In [ ]:
def show_video(video_path, trim=False, duration=10, width=800):
    """
    Convert (and optionally trim) a video, suppress ffmpeg output, and display it inline.
    Parameters:
        video_path (str): Path to the input video file.
        trim (bool): Whether to trim the video to a short preview (default False).
        duration (int): Duration in seconds if trimming (default 10).
        width (int): Display width in pixels (default 800).
    """
    output_path = "preview.mp4"

    # Build ffmpeg command
    cmd = ["ffmpeg", "-i", video_path]
    if trim:
        cmd += ["-t", str(duration)]
    cmd += [output_path, "-y"]  # overwrite existing

    # Run ffmpeg silently (no stdout/stderr)
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Display the video inline
    display(Video(output_path, width=width, embed=True))

if __name__ == "__main__":
    show_video("videos/normanniles3_2025-10-26-17-50-45.mp4")

In [ ]:
counter = 0
for video in os.listdir(video_path):
    full_path = os.path.join(video_path, video)
    show_video(full_path)  # display the full video
    #show_video("normanniles1_2025-10-20-06-01-45.mp4", trim=True)     # 10s preview
    #show_video("normanniles1_2025-10-20-06-01-45.mp4", trim=True, duration=5)  # 5s preview
    counter += 1
    if counter == 2:
        break

# Mediapipe object detection setup

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import os
from google.colab.patches import cv2_imshow
import pandas as pd # Import pandas for DataFrame creation
import re # Import regex for timestamp extraction


model_path = '/absolute/path/to/lite-model_efficientdet_lite0_detection_metadata_1.tflite'

# Populate congestion_map from the train DataFrame
congestion_map = {}
for index, row in train.iterrows():
    # The 'videos' column in train DataFrame contains paths like 'normanniles1/normanniles1_2025-10-20-10-04-45.mp4'
    # os.path.basename extracts 'normanniles1_2025-10-20-10-04-45.mp4'
    video_full_path = row['videos']
    video_filename_base = os.path.basename(video_full_path)
    congestion_map[video_filename_base] = {
        'congestion_enter_rating': row['congestion_enter_rating'],
        'congestion_exit_rating': row['congestion_exit_rating']
    }

BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path='/content/efficientdet.tflite', delegate=BaseOptions.Delegate.GPU),
    max_results=-1,
    score_threshold=0.5,
    running_mode=VisionRunningMode.VIDEO,
    category_allowlist= ["person", "truck", "car", "motorcycle", "bus"],
    )

# Initialize detector ONCE outside the loop
detector = ObjectDetector.create_from_options(options)

all_video_results = [] # List to store results for all videos
cumulative_timestamp_ms = 0 # Initialize a global cumulative timestamp

# Get list of video files and sort them by timestamp
def extract_timestamp_from_filename(filename):
    match = re.search(r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}', filename)
    if match:
        return match.group(0)
    return filename # Return original name if no timestamp found, for stable sorting

video_files_in_dir = [f for f in os.listdir(video_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
sorted_video_files = sorted(video_files_in_dir, key=extract_timestamp_from_filename)

try:
    for video_file_name in sorted_video_files:
        # Skip non-video files like .ipynb_checkpoints
        if not video_file_name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            print(f"Skipping non-video file: {video_file_name}")
            continue

        # Initialize objects dictionary for each video
        objects = {"car_left":0,"car_right":0,"truck_left":0,"truck_right":0,"motorcycle_left":0,"motorcycle_right":0,"person_left":0,"person_right":0,"bus_left":0,"bus_right":0}

        # Retrieve congestion ratings
        congestion_info = congestion_map.get(video_file_name, {'congestion_enter_rating': 'unknown', 'congestion_exit_rating': 'unknown'})
        enter_rating = congestion_info['congestion_enter_rating']
        exit_rating = congestion_info['congestion_exit_rating']

        full_path = os.path.join(video_path, video_file_name)
        cap = cv2.VideoCapture(full_path)
        video_file_fps = cap.get(cv2.CAP_PROP_FPS)
        frame_duration_ms = int(1000 / video_file_fps) if video_file_fps > 0 else 0 # Calculate duration of one frame for this video

        if not cap.isOpened():
            print(f"Error: Could not open video file {full_path}")
            continue

        print(f"Processing video: {video_file_name}")
        frame_index_within_video = 0 # Keeping this for local count if needed for other logic
        while True: # Loop until break
            ret, frame = cap.read()
            if ret:
                # Get frame dimensions
                frame_height, frame_width, _ = frame.shape
                frame_midpoint_x = frame_width / 2
            else: # Use else for consistency with if not ret
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            # Use the global cumulative timestamp
            detection_result = detector.detect_for_video(mp_image, cumulative_timestamp_ms)

            # Increment the global cumulative timestamp for the next frame
            cumulative_timestamp_ms += frame_duration_ms

            for detection in detection_result.detections:
                bbox = detection.bounding_box

                category_name = detection.categories[0].category_name if detection.categories else "Unknown"
                score = round(detection.categories[0].score, 2) if detection.categories else 0.0

                # Calculate object's horizontal center for every detection
                object_center_x = bbox.origin_x + bbox.width / 2
                # Classify as 'left' or 'right' for every detection
                position_label = "left" if object_center_x < frame_midpoint_x else "right"

                if str(category_name) in["person", "truck", "car", "motorcycle", "bus"] and score >= 0.57:
                    # Update corresponding count only if criteria met
                    object_key = f'{category_name}_{position_label}'
                    if object_key in objects: # Ensure key exists before incrementing
                        objects[object_key] += 1

            # cv2_imshow(annotated_frame) # Display the annotated frame (commented out in original, keeping it that way)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            frame_index_within_video += 1 # Increment local frame index
        print(f"Objects detected in {video_file_name}: {objects}")
        print(f"Congestion Enter Rating: {enter_rating}, Congestion Exit Rating: {exit_rating}")

        # Store results for the current video
        video_data = {
            'video_filename': video_file_name,
            'congestion_enter_rating': enter_rating,
            'congestion_exit_rating': exit_rating,
        }
        video_data.update(objects) # Add object counts to the dictionary
        all_video_results.append(video_data)

        cap.release() # Release the capture object after processing each video
        cv2.destroyAllWindows() # Close any OpenCV windows (important for local execution environments)
finally:
    # Close the detector after all videos are processed
    detector.close()

# Convert the list of dictionaries to a Pandas DataFrame
results_df = pd.DataFrame(all_video_results)
print("\n--- Results DataFrame ---")
display(results_df.head())


In [23]:
output_csv_path = 'video_analysis_results.csv'
results_df.to_csv(output_csv_path, index=False)
print(f"Results saved to {output_csv_path}")

Results saved to video_analysis_results.csv


# Objects counter

In [ ]:
"""
generate an initial counts sheet for the objects by setting ranges for the
number of objects and the classes in the training data and classify
the test data based on them.

check for how to count the accurcy and f1-score
"""

In [3]:
import pandas as pd # Ensure pandas is imported
import os

# Define the directory where results CSVs are located
results_directory = '/content/data'

all_results_df = []

# Iterate over all files in the specified directory
print(f"Searching for CSV files in: {results_directory}")
if os.path.exists(results_directory):
    for filename in os.listdir(results_directory):
        if filename.endswith('.csv') and filename.startswith('video_analysis_results_'):
            file_path = os.path.join(results_directory, filename)
            try:
                df = pd.read_csv(file_path)
                all_results_df.append(df)
                print(f"Loaded {len(df)} rows from '{file_path}'")
            except Exception as e:
                print(f"Error loading '{file_path}': {e}")
else:
    print(f"Error: Directory '{results_directory}' not found.")

if all_results_df:
    results_df = pd.concat(all_results_df, ignore_index=True)
    print(f"\nSuccessfully combined {len(all_results_df)} files into a single DataFrame with {len(results_df)} total rows.")
    display(results_df.head())
else:
    print("Error: No results CSV files found or loaded from the specified directory. Cannot proceed with analysis.")
    results_df = pd.DataFrame() # Initialize an empty DataFrame to prevent NameError


Searching for CSV files in: /content/data
Loaded 39 rows from '/content/data/video_analysis_results_12030_12070.csv'
Loaded 10 rows from '/content/data/video_analysis_results_4015_4025.csv'
Loaded 98 rows from '/content/data/video_analysis_results_200_300.csv'
Loaded 100 rows from '/content/data/video_analysis_results_0_100.csv'

Successfully combined 4 files into a single DataFrame with 247 total rows.


,video_filename,congestion_enter_rating,congestion_exit_rating,car_left,car_right,truck_left,truck_right,motorcycle_left,motorcycle_right,person_left,person_right,bus_left,bus_right,video_file_name
0,normanniles4_2025-10-20-06-00-45.mp4,free flowing,free flowing,4,19,12,2,0,0,0,0,0,0,NaN
1,normanniles4_2025-10-20-06-01-45.mp4,free flowing,free flowing,43,0,0,0,0,0,0,0,0,0,NaN
2,normanniles4_2025-10-20-06-02-45.mp4,free flowing,free flowing,0,7,0,0,0,0,0,0,0,0,NaN
3,normanniles4_2025-10-20-06-03-45.mp4,free flowing,free flowing,49,60,0,0,0,0,0,0,0,0,NaN
4,normanniles4_2025-10-20-06-04-45.mp4,free flowing,free flowing,33,20,0,0,0,0,0,0,0,0,NaN


In [5]:
import re

# Extract camera_id from video_filename
# Assuming filenames are like 'normanniles3_2025-10-26-17-33-45.mp4'
# Convert column to string type first to handle potential non-string entries (e.g., NaN) gracefully
results_df['camera_id'] = results_df['video_filename'].astype(str).apply(lambda x: re.match(r'([a-zA-Z0-9]+)', x).group(1) if re.match(r'([a-zA-Z0-9]+)', x) else 'unknown')

# Define a numerical mapping for congestion ratings
congestion_mapping = {
    'free flowing': 0,
    'light delay': 1,
    'moderate delay': 2,
    'heavy delay': 3,
    'very heavy delay': 4,
    'unknown': -1 # Assign a distinct value for unknown/unclassified ratings
}

# Apply the numerical mapping to 'congestion_enter_rating'
results_df['congestion_enter_rating_numeric'] = results_df['congestion_enter_rating'].map(congestion_mapping)

# Calculate mean and variance of congestion_enter_rating for each camera_id
congestion_stats = results_df.groupby('camera_id')['congestion_enter_rating_numeric'].agg(['mean', 'var']).reset_index()

# Display the results
print("\nMean and Variance of Congestion Enter Rating by Camera ID:")
display(congestion_stats)



Mean and Variance of Congestion Enter Rating by Camera ID:


,camera_id,mean,var
0,nan,0.000000,0.000000
1,normanniles1,1.254902,1.538342
2,normanniles2,0.000000,0.000000
3,normanniles3,0.923077,0.873846
4,normanniles4,0.000000,0.000000


In [6]:
# Define the columns that represent vehicle counts
vehicle_columns = [
    'car_left', 'car_right',
    'truck_left', 'truck_right',
    'motorcycle_left', 'motorcycle_right',
    'bus_left', 'bus_right'
]

# Ensure these columns exist in results_df before attempting to sum them
# Filter out any columns that might be missing if a video had no detections for them
existing_vehicle_columns = [col for col in vehicle_columns if col in results_df.columns]

if not existing_vehicle_columns:
    print("Warning: No vehicle count columns found in results_df. Please check the object detection output.")
    # Initialize total_vehicles to 0 if no vehicle columns are present
    results_df['total_vehicles'] = 0
else:
    # Sum all vehicle counts for each video
    results_df['total_vehicles'] = results_df[existing_vehicle_columns].sum(axis=1)

print("Total vehicles calculated per video. Displaying head with new column:")
display(results_df[['video_filename', 'congestion_enter_rating', 'total_vehicles']].head())

Total vehicles calculated per video. Displaying head with new column:


,video_filename,congestion_enter_rating,total_vehicles
0,normanniles4_2025-10-20-06-00-45.mp4,free flowing,37
1,normanniles4_2025-10-20-06-01-45.mp4,free flowing,43
2,normanniles4_2025-10-20-06-02-45.mp4,free flowing,7
3,normanniles4_2025-10-20-06-03-45.mp4,free flowing,109
4,normanniles4_2025-10-20-06-04-45.mp4,free flowing,53


In [10]:
# Calculate mean and variance of total_vehicles for each camera_id and congestion rating
# We'll use the numeric congestion rating for proper statistical calculation
vehicle_congestion_stats = results_df.groupby(['camera_id', 'congestion_enter_rating_numeric']).agg(
    mean=('total_vehicles', 'mean'),
    var=('total_vehicles', 'var'),
    count=('total_vehicles', 'count'),
    min_observed_vehicles=('total_vehicles', 'min'), # Get the min observed value
    max_observed_vehicles=('total_vehicles', 'max')  # Get the max observed value
).reset_index()

# Map back the original congestion rating for readability
# Make sure congestion_mapping is available (from cell 90ce029b)
if 'congestion_mapping' in locals():
    reverse_congestion_mapping = {v: k for k, v in congestion_mapping.items()}
    vehicle_congestion_stats['congestion_enter_rating_label'] = vehicle_congestion_stats['congestion_enter_rating_numeric'].map(reverse_congestion_mapping)

# Calculate standard deviation
vehicle_congestion_stats['std_dev'] = vehicle_congestion_stats['var'].apply(lambda x: x**0.5)

# Calculate integer ranges (mean +/- 1 standard deviation, ensuring non-negative lower bound)
vehicle_congestion_stats['lower_bound_vehicles'] = (vehicle_congestion_stats['mean'] - vehicle_congestion_stats['std_dev']).round().astype(int)
vehicle_congestion_stats['lower_bound_vehicles'] = vehicle_congestion_stats['lower_bound_vehicles'].apply(lambda x: max(0, x)) # Ensure lower bound is not negative
vehicle_congestion_stats['upper_bound_vehicles'] = (vehicle_congestion_stats['mean'] + vehicle_congestion_stats['std_dev']).round().astype(int)

# --- Now, find the filenames corresponding to the min and max observed vehicles ---
# Get the index of the minimum total_vehicles for each group
idx_min = results_df.groupby(['camera_id', 'congestion_enter_rating_numeric'])['total_vehicles'].idxmin()
min_video_filenames = results_df.loc[idx_min, ['camera_id', 'congestion_enter_rating_numeric', 'video_filename']]
min_video_filenames = min_video_filenames.rename(columns={'video_filename': 'min_vehicles_file'})

# Get the index of the maximum total_vehicles for each group
idx_max = results_df.groupby(['camera_id', 'congestion_enter_rating_numeric'])['total_vehicles'].idxmax()
max_video_filenames = results_df.loc[idx_max, ['camera_id', 'congestion_enter_rating_numeric', 'video_filename']]
max_video_filenames = max_video_filenames.rename(columns={'video_filename': 'max_vehicles_file'})

# Merge these filenames back into the vehicle_congestion_stats DataFrame
vehicle_congestion_stats = pd.merge(
    vehicle_congestion_stats, min_video_filenames,
    on=['camera_id', 'congestion_enter_rating_numeric'], how='left'
)
vehicle_congestion_stats = pd.merge(
    vehicle_congestion_stats, max_video_filenames,
    on=['camera_id', 'congestion_enter_rating_numeric'], how='left'
)


print("\nMean, Variance, and Count of Total Vehicles by Camera ID and Congestion Rating (with calculated ranges and extremal files):")
display(vehicle_congestion_stats.sort_values(by=['camera_id', 'congestion_enter_rating_numeric']))


Mean, Variance, and Count of Total Vehicles by Camera ID and Congestion Rating (with calculated ranges and extremal files):


,camera_id,congestion_enter_rating_numeric,mean,var,count,min_observed_vehicles,max_observed_vehicles,congestion_enter_rating_label,std_dev,lower_bound_vehicles,upper_bound_vehicles,min_vehicles_file,max_vehicles_file
0,nan,0,38.970000,1696.837475,100,0,225,free flowing,41.192687,0,80,NaN,NaN
1,normanniles1,0,1294.348837,257835.137320,43,253,2173,free flowing,507.774691,787,1802,normanniles1_2025-10-20-10-05-45.mp4,normanniles1_2025-10-20-11-16-45.mp4
2,normanniles1,1,1413.533333,199790.838095,15,468,1959,light delay,446.979684,967,1861,normanniles1_2025-10-20-10-04-45.mp4,normanniles1_2025-10-20-11-10-45.mp4
3,normanniles1,2,1832.947368,127140.497076,19,1119,2570,moderate delay,356.567661,1476,2190,normanniles1_2025-10-26-17-58-45.mp4,normanniles1_2025-10-20-11-31-45.mp4
4,normanniles1,3,1659.440000,49863.506667,25,1280,2113,heavy delay,223.301381,1436,1883,normanniles1_2025-10-20-11-29-45.mp4,normanniles1_2025-10-20-10-30-45.mp4
5,normanniles2,0,43.500000,982.300000,6,0,84,free flowing,31.341666,12,75,normanniles2_2025-10-20-06-03-45.mp4,normanniles2_2025-10-20-06-02-45.mp4
6,normanniles3,0,487.200000,14239.955556,10,277,678,free flowing,119.331285,368,607,normanniles3_2025-10-26-17-52-45.mp4,normanniles3_2025-10-26-17-38-45.mp4
7,normanniles3,1,646.100000,36891.433333,10,375,939,light delay,192.071428,454,838,normanniles3_2025-10-26-17-33-45.mp4,normanniles3_2025-10-26-17-47-45.mp4
8,normanniles3,2,849.500000,254881.666667,4,350,1548,moderate delay,504.858066,345,1354,normanniles3_2025-10-26-17-50-45.mp4,normanniles3_2025-10-26-17-49-45.mp4
9,normanniles3,3,806.000000,130050.000000,2,551,1061,heavy delay,360.624458,445,1167,normanniles3_2025-10-26-17-53-45.mp4,normanniles3_2025-10-26-17-48-45.mp4


# Objects dataset

In [ ]:
"""
video_list = []

for images in in the video:
    detect the most important 15 objects for each frame
    video_list.append(objects)

with open (video):
    create a csv file for the video with the objects name and score
"""

# Frame classification setup


In [ ]:
"""
build a Keras model to classify the frames into the required classes for the density of traffic
then find the most common to get the density of traffic for the full video.
"""